### Read & Explore

In [0]:
from pyspark.sql.functions import trim, col, when

# Read sales_details from Bronze
df = spark.read.table("`databricks-medallion-lakehouse`.bronze.sales_details")

print("="*70)
print("SILVER: sales_details Transformation")
print("="*70)
print(f"\nBronze rows: {df.count():,}")

print("\nData types (notice dates are INTEGER, not DATE):")
df.printSchema()

print("\nFirst 3 rows (BEFORE cleaning):")
df.show(3, truncate=False)

### Convert Dates from Integer to DATE

In [0]:
from pyspark.sql.functions import try_to_date, lpad, col

# Step 1: Convert integer dates SAFELY (handle invalid dates)
df_clean = df.select(
    col("sls_ord_num"),
    col("sls_prd_key"),
    col("sls_cust_id"),
    # Use try_to_date: converts invalid dates to NULL instead of failing
    try_to_date(lpad(col("sls_order_dt").cast("string"), 8, "0"), "yyyyMMdd").alias("sls_order_dt"),
    try_to_date(lpad(col("sls_ship_dt").cast("string"), 8, "0"), "yyyyMMdd").alias("sls_ship_dt"),
    try_to_date(lpad(col("sls_due_dt").cast("string"), 8, "0"), "yyyyMMdd").alias("sls_due_dt"),
    col("sls_sales"),
    col("sls_quantity"),
    col("sls_price")
)

print("Invalid dates converted to NULL:")
print(f"  NULL order_dates: {df_clean.filter(col('sls_order_dt').isNull()).count()}")
print(f"  NULL ship_dates: {df_clean.filter(col('sls_ship_dt').isNull()).count()}")
print(f"  NULL due_dates: {df_clean.filter(col('sls_due_dt').isNull()).count()}")

print("\nFirst 3 rows (dates now safe):")
df_clean.show(3, truncate=False)

### Rename Columns

In [0]:
# Step 2: Rename columns to professional names
df_clean = df_clean.select(
    col("sls_ord_num").alias("order_number"),
    col("sls_prd_key").alias("product_key"),
    col("sls_cust_id").alias("customer_id"),
    col("sls_order_dt").alias("order_date"),
    col("sls_ship_dt").alias("ship_date"),
    col("sls_due_dt").alias("due_date"),
    col("sls_sales").alias("sales_amount"),
    col("sls_quantity").alias("quantity"),
    col("sls_price").alias("price")
)

print("Final schema:")
df_clean.printSchema()

print("\nFirst 3 rows (FINAL):")
df_clean.show(3, truncate=False)

### Write to Silver

In [0]:
# Step 3: Deduplicate by order_number (each order should be unique)
df_clean = df_clean.dropDuplicates(["order_number"])

rows_final = df_clean.count()
print(f"\nFinal rows: {rows_final:,}")

# Step 4: Write to Silver
silver_table = "`databricks-medallion-lakehouse`.silver.sales_details"

spark.sql(f"DROP TABLE IF EXISTS {silver_table}")

df_clean.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(silver_table)

print(f"✅ Written to Silver: {silver_table}")